> Crop image based on different criteria

In [ ]:
#| default_exp crop_image_center

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
from fastcore.all import *
from tqdm.auto import tqdm
from fastcore.basics import *
from PIL import Image
from typing import Union
import numpy as np
import sys
import cv2
from argparse import ArgumentParser

In [ ]:
import matplotlib as mpl
mpl.rcParams['image.cmap']='gray'

In [ ]:
#| export
path=Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/2023_easy_pin_detection/nbs')
os.chdir(path)

In [ ]:
#| export
custom_lib_path=Path(r'/home/ai_warstein/custom_libs/goni/custom_libs')
sys.path.append(str(custom_lib_path))

In [ ]:
os.getenv('PYTHONPATH')

In [ ]:
os.getenv('PATH')

In [ ]:
#| export
from dotenv import load_dotenv

In [ ]:
#| export
dotenv_p = load_dotenv(Path(Path(path).parent, 'private_easy_pin_detection/.env'))
CV_TOOLS = os.getenv('CV_TOOLS')
PRIVATE_EASY_PIN_DETECTION = os.getenv('PRIVATE_EASY_PIN_DETECTION')
sys.path.append(CV_TOOLS)
sys.path.append(PRIVATE_EASY_PIN_DETECTION)

In [ ]:
#| export
from cv_tools.core import *
from cv_tools.cv_ops import *

In [ ]:
#| exporti
from private_easy_pin_detection.core import *

In [ ]:
#| export
def pil_img(im_path:str):
    if im_path is not None:
        return read_img(im_path, cv=False, gray=False)
    else:
        return None

In [ ]:
#| export
class CropImage:
    def __init__(self,
                 im_path:Path, # PIL Image
                 im1_path:Path, # Important to get the reference image rotation
                 ref_im_path:str, # reference image path
                 im_width:int,
                 im_height:int,
                 im_save_path:Path,
                 c_crop:bool=False, # whether to use center crop
                 rotate_:bool=True, # whether to rotate the image based on reference image
                 msk_path:Path=None, # PIL Image
                 msk_save_path:Path=None,
                 ) -> None:
        # with cast=True, pil_img function(type annotation) will be applied to the the init arguments
        store_attr()
    __repr__ = basic_repr('im_path, msk_path, im_width, im_height, im_save_path, msk_save_path')

In [ ]:
# |export
@patch
def center_crop_(self:CropImage, 
                img:Image, 
                msk:Image=None):

    crop_im = center_crop(img,
                desired_width=self.im_width,
                desired_height=self.im_height,
                height_offset=0,
                width_offset=0)

    if self.msk_path is not None:
        crop_msk = center_crop(msk,
                    desired_width=self.im_width,
                    desired_height=self.im_height,
                    height_offset=0,
                    width_offset=0)
    else:
        crop_msk = None
    return crop_im, crop_msk



In [ ]:
# |export
@patch
def center_crop_v02(self:CropImage, 
                img:Image, 
                msk:Image=None):

    crop_im = center_crop(img,
                desired_width=self.im_width,
                desired_height=self.im_height,
                height_offset=0,
                width_offset=95)

    if self.msk_path is not None:
        crop_msk = center_crop(msk,
                    desired_width=self.im_width,
                    desired_height=self.im_height,
                    height_offset=0,
                    width_offset=0)
    else:
        crop_msk = None
    return crop_im, crop_msk



In [ ]:
#| export
@patch
def crop_vision_pc(self:CropImage,img:Image, msk:Image=None):
        # Get the current size of the image
        width, height = img.size
        #center_x = (width // 2 -50)
        # because vision framework fixed center_x and center_y
        center_x = (width // 2 -6.5) # 1020.5
        center_y = (height // 2 -58.5) # 1169.5


        # Calculate the coordinates of the top-left corner of the crop
        left = center_x - self.im_width // 2
        top = center_y - self.im_height // 2

        # Calculate the coordinates of the bottom-right corner of the crop
        right = center_x + self.im_width // 2
        bottom = center_y + self.im_height // 2

        # Crop the image
        cropped_image = img.crop((left, top, right, bottom))
        if self.msk_path is not None:
            cropped_mask = np.array(msk.crop((left, top, right, bottom)))
        else:cropped_mask = None
        #print(f'cropped_image size = {cropped_image.size}')
        return np.array(cropped_image), cropped_mask



In [ ]:
#| export
@patch
def read_reference_image(
    self:CropImage, 
    fn:str, # image file name
    )->np.ndarray:

    ref_im_name = ref_fn_frm_fn(fn)
    ref_im_fn = Path(self.ref_im_path, ref_im_name)


    return read_img(f'{ref_im_fn}')

In [ ]:
#| export
@patch
def get_rotation_angle(
    self:CropImage,
    ref_img:np.ndarray, # reference image
    tst_img:np.ndarray, # test image which needs to be rotated
    method:str='sift' # method to find the rotation angle
    )->float:# Rotated image, rotation angle
    angle_ = find_rotation_angle(ref_img, tst_img, method=method)
    if angle_ >5:
        angle_n = find_rotation_angle(ref_img, tst_img, method='orb')
        if angle_n >5:
            angle_n = 1.01
            return angle_n
        else:return angle_n
    else:return angle_

In [ ]:
#| export
@patch
def get_image1_name(
    self:CropImage, 
    fn:str)->str:

    return fn.replace('image2', 'image1').replace('processed_image', 'image1')

In [ ]:
#| export
@patch
def __call__(self:CropImage):

    if self.im_save_path is not None:
        Path(self.im_save_path).mkdir(exist_ok=True,parents=True)

    if self.msk_save_path is not None:
        Path(self.msk_save_path).mkdir(exist_ok=True, parents=True)

    images = Path(self.im_path).ls()
    im_names = get_name_(images)
    for i in tqdm(im_names,total=len(im_names)):
        img = pil_img(f'{self.im_path}/{i}')
        try:
            if self.rotate_:
                im1_name = self.get_image1_name(i)
                img1_img = read_img(f"{Path(self.im1_path, im1_name)}")

                # Getting and reading reference image based on file name
                ref_img = self.read_reference_image(i)
                # Image was pil, thats why np.array is used to convert it to numpy array
                angle_ = self.get_rotation_angle(ref_img, img1_img)
                img = rotate_image(np.array(img), angle_, interpolation_method=cv2.INTER_LINEAR)
                img = Image.fromarray(img)
        except Exception as e:
            print(e)
            continue


        if self.msk_path is not None:
            msk = pil_img(f'{self.msk_path}/{i}')
            if self.rotate_:
                msk = rotate_image(np.array(msk), angle_, interpolation_method=cv2.INTER_LINEAR)
                msk = Image.fromarray(msk)
        else:
            msk = None
        if self.c_crop:
            cr_im, cr_msk = self.center_crop_v02(img, msk)
            if self.im_path is not None:
                cv2.imwrite(f'{self.im_save_path}/{i}', cr_im)
            if cr_msk is not None:
                cv2.imwrite(f'{self.msk_save_path}/{i}', cr_msk)
        else:
            cr_im, cr_msk = self.crop_vision_pc(img, msk)
            if self.im_path is not None:
                cv2.imwrite(f'{self.im_save_path}/{i}', cr_im)
            if cr_msk is not None:
                cv2.imwrite(f'{self.msk_save_path}/{i}', cr_msk)

In [ ]:
im_path=Path(r'/home/ai_sintercra.work/Fail_investigation/additional/FailSaved20240709/FailSaved20240709_unzip/main_im2')
ref_im_path=Path(r'/home/ai_easypid.work/Reference_images')
im1_path=Path(Path(im_path).parent, 'main_im1')
im_save_path=Path(Path(im_path).parent, 'main_im2_cropped')
c_crop=True
msk_save_path=None
msk_path=None
im_width=1632
im_height=1152
crop_ = CropImage(
        im_path=im_path,
        im1_path=im1_path,
        ref_im_path=ref_im_path,
        im_save_path=im_save_path,
        c_crop=c_crop,
        msk_save_path=msk_save_path,
        msk_path=msk_path,
        im_width=im_width,
        im_height=im_height,
        rotate_=True
    )

In [ ]:
crop_()

In [ ]:
#| export
def parse_args_():
    parser = ArgumentParser()
    parser.add_argument(
        '--im_path',
        type=str,
        help='path to the image directory',
        required=True
    )
    parser.add_argument(
        '--im1_path',
        type=str,
        help='path to the image directory of image1',
        required=True
    )
    parser.add_argument(
        '--ref_im_path',
        type=str,
        help='path to the reference image directory',
        required=False,
        default='/home/ai_easypid.work/Reference_images'
    )
    parser.add_argument(
        '--msk_path',
        type=str,
        help='path to the mask directory',
        default=None
    )
    parser.add_argument(
        '--im_width',
        type=int,
        help='desired width of the image',
        required=True
    )
    parser.add_argument(
        '--im_height',
        type=int,
        help='desired height of the image',
        required=True
    )
    parser.add_argument(
        '--im_save_path',
        type=str,
        help='path to save the cropped images',
        default=None

    )
    parser.add_argument(
        '--msk_save_path',
        type=str,
        help='path to save the cropped images',
        default=None

    )
    parser.add_argument(
        '--c_crop',
        action='store_true',
        help='whether to use center crop',
        default=False
    )
    parser.add_argument(
        '--rotate',
        action='store_true',
        help='whether to rotate the image based on reference image',
        default=False
    )
    return parser.parse_args()

In [ ]:
#| export
def main():
    args = parse_args_()

    crop_ = CropImage(
        im1_path=args.im1_path,
        im_path=args.im_path,
        ref_im_path=args.ref_im_path,
        im_save_path=args.im_save_path,
        c_crop=args.c_crop,
        msk_save_path=args.msk_save_path,
        msk_path=args.msk_path,
        im_width=args.im_width,
        im_height=args.im_height,
        rotate_=args.rotate
    )
    crop_()
    

In [ ]:
#| export
if __name__ == '__main__':
    main()

In [ ]:
from pathlib import Path

In [ ]:
Path.cwd()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export('n:/homes/hasan/projects/git_data/2023_easy_pin_detection/nbs/16_crop_image.ipynb')